# Fine Tuning.py

This notebook contains all of the code necessary to fine-tune a large language model for either search retrieval or reranking of search results. It can be used to fine-tune either off-the-shelf models (loaded directly from Hugging Face) or models that have already completed Domain Adaptation via TSDAE. Broadly, fine-tuning works in three main steps:

* First, training data is loaded and prepared for analysis. The `sentence-transformers` package requires training data to be in the form of a `Dataset` object, which is a row-based structure that uses a list of column names to track the various fields of each row of the dataset, represented by a list. There is one list / row per "example" in the dataset, and particular column-fields of a row of data can be accessed using the name of the column in the `Dataset`'s dictionary. The format the data needs to be in depends on the type of fine-tuning you want to perform (e.g. what algorithm to employ, what loss function to use, was hard negative mining performed, etc.).
* Second, the model to-be-trained is loaded in the appropriate format. For search models (used for retrieving approximate nearest neighbor results), this format is a `SentenceTransformer` object so that bidirectional encoding and decoding can be used. For reranker models, this format is a `CrossEncoder` object, so that text strings can be directly compared to one another.
* Third, the parameters that govern fine-tuning are initialized and passed to a `Trainer` object. This is where properties such as learning rate and number of training epochs are specified. 

Once these steps are completed, you can run fine-tuning to produce a model with updated internal weights and hidden states that should be better suited to your data task. Each of the above steps, as well as the procedure overall, is discussed in more detail in the relevant sections below.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

Additionally, the `Datasets` package used to prepare and batch the TSDAE examples requires a particular optimization back-end; we need to pin specific versions here so that we can get the right compatibility.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec datasets sentence-transformers --upgrade torch --upgrade transformers --upgrade accelerate --upgrade

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Authenticate to Key Vault
credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

# Define the workspace subscription and resources so we can instantiate a secure client
SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

# NOTE: Even though we're not directly calling any of the client functions, we do still need
# the object. Having a client instantiated acts as an authenticated connection for our compute
# instance to connect to the workspace.
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Finally, performing fine-tuning is only feasible with a compute instance attached to GPU. This cell ensures that the GPU is available for CUDA optimization.

In [ ]:
import torch
assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC storage container. Any `.txt` or `.csv` files just need to be uploaded to the desired directory within the DIBBs storage container. Unlike with more complex file types, there is no need to manually turn these files into Azure Data Assets before using them. They can simply be directly loaded from storage once they're uploaded.

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Load and Prepare Tuning Data

Using our file mount, we can read a list of positive pairs from Azure Blob Storage to use as training data. We decode the bytestrings into proper UTF-8, then read all example pairs into a list for some preprocessing. First, we'll randomize the examples, and then we'll select a sub-set of them for fine-tuning, based on a given size. How we format the data after that depends on how we want to structure fine-tuning. There are multiple ways we can organize the data, but all rely on a few common concepts:

* _Anchor Codes_ are correct standardized LOINCs that will represent to the model what the "ideal" of each "class" will look like (for Text to Code, standardized LOINC name variants are effectively hundreds of thousands of different classes, and the task of standardizing narrative free-text input amounts to mapping unknown examples to the correct class)
* _Positive Codes_ are nonstandard, free-text inputs that belong to the class defined by an anchor code, even though they vary from the correct standardization. All of the synthetic training examples the TtC team produced to emulate production data are effectively positive codes. We want the model to learn to associate these nonstandard codes with the "ideal" definition of each class.
* _Negative Codes_ are a product of Hard Negative Mining (described in another notebook). For a given anchor code, a negative code is a narrative free-text input that "looks like" the anchor, but does not belong to the class the anchor represents. Negative Codes aren't used in all forms of fine-tuning (notably, search retriever training eschews them), but they are important for teaching reranker models how to separate near-matches from true matches.
* _Labels_ are simply 1s and 0s present in some data rows denoting whether a code is a positive or negative, respectively.

These core concepts are used across a number of different ways of organizing data. This notebook supports 6 different ways of structuring data for fine-tuning. For retriever training, only one way (AP-Pairs) is used, which is why we can load the data from a text file source instead of requiring loading from a `sentence-transformers` `Dataset`. However, for reranker training, any of the 6 forms of organization can be chosen, so long as the data format is paired with an appropriate loss function. The 6 supported dataset formats are:

* **AP Pairs:** Anchor-Positive pairs are the simplest form of training data we can provide to fine-tuning. They have the form `(anchor, positive)`, and they do not have a `label` column. AP Pairs are compatible only with MNRL Loss.
* **APN Pairs:** Anchor Positive/Negative pairs are slightly more complicated because the second element can be a positive or negative code. They have the form `(anchor, positive/negative)`, and must be accompanied by a `label` column that is `1` if the second element is positive and `0` if the second element is negative. APN Pairs support Contrastive and Online Contrastive loss functions.
* **Scored Pairs:** Scored pairs have the exact same structure as APN pairs, but instead of a binary `label`, they instead have a float similarity `score` column denoting how similar the second element should be to the anchor. Scored Pairs are compatible with CoSENT and Angle losses.
* **APN Triplets:** APN triplets are three-tuples of the form `(anchor, positive, negative)` that present each anchor code row with both a positive and a negative member of the class. APN Triplets do not need to be accompanied by a `label` column. They can be used with MNRL and Triplet loss.
* **APN List:** APN Lists are akin to "extended" forms of APN Triplets. Instead of having one positive and one negative associated with each anchor, APN Triplets take the form `(anchor, positive, negative_1, ..., negative_k)`, where `k` can be a user-chosen number (likely chosen during hard negative mining). APN Lists are compatible only with MNRL loss.
* **Labeled List:** Labeled Lists are a unique form of data organization in that their fields are themselves lists. Labeled List is the default output of data produced by hard negative mining, meaning this is also the format that data is saved to and loaded from when working with HNM Dataset files. Labeled Lists have the form `(anchor, [input_1, input_2, ..., input_k], [label_1, label_2, ..., label_k])`, where as above, `k` is a number decided during negative mining. Within the `input` and `label` lists, there must be the same number of elements, and each label must describe the input at the same index in the other list. One input _must_ be positive (typically the first element), but the rest should be negative. Labeled Lists are compatible only with the Lambda Loss function.

There is no universally right answer for which data format to select. As with most of data science, the best performing choice must be determined through experimentation. During our testing, we found that AP-Pairs performed best for training search retriever models. For training rerankers, we found that any of these options can be performant, though anecdotally, Labeled Lists with Lambda Loss frequently did well.

In any case, after data is appropriately loaded and partitioned, we create a Dataset object that `sentence-transformers` can read during training batching. This also lets us split out a validation set that we can use to monitor loss during training.


In [ ]:
import random
from datasets import Dataset, load_from_disk

# NOTE: If you're training a Retriever (for semantic search), this MUST be set to
# False. If you're training a Reranker, and are therefore working with data created
# using hard negative mining, this MUST be set to True.
LOADING_DIRECTLY_SAVED_DATASET = True

FINE_TUNING_DATA_FILE = "hnm_prod_emulated_reranker_pairs_no_cn_1_5_margin_03"
TRAINING_SIZE = 220_000
VALIDATION_SIZE = 21_690
RANDOM_SEED = 1246

# DESIRED DATA FORMAT
# ap_pairs: (anchor, positive), no label; MNRL
# apn_pairs: (anchor, positive/negative), label 1 if positive 0 if negative; contrastive, online contrastive
# scored_pairs: (anchor, positive/negative), float similarity score; cosent, angle
# apn_triplets: (anchor, positive, negative), no label; MNRL, triplet
# apn_list: (anchor, positive, neg_1, neg_2, ...), no label; MNRL
# labeled_list: (anchor, [pos_1, ..., pos_k], [label_1, ..., label_k]); Lambda
DATA_FORMAT = "ap_pairs"
NEGATIVES_PER_DATASET_ROW = 5

if LOADING_DIRECTLY_SAVED_DATASET:
    # First, make sure we're using allowed values
    assert DATA_FORMAT in [
    "ap_pairs", "apn_pairs", "scored_pairs", "apn_triplets", "apn_list", "labeled_list"
    ]
    assert NEGATIVES_PER_DATASET_ROW >= 1

    dataset = load_from_disk(FINE_TUNING_DATA_FILE)
    ds_columns = dataset.column_names

    # Standardize column names if necessary
    if "query" in ds_columns:
        dataset = dataset.rename_column("query", "anchor")
    if "docs" in ds_columns:
        datset = dataset.rename_column("docs", "positive")
    if "scores" in ds_columns:
        dataset = dataset.rename_column("scores", "labels")


    if DATA_FORMAT == "ap_pairs":
        print("Formatting data as Anchor-Positive Pairs")
        def _map_positive(example):
            # The first element in a dataset row is always the positive instance,
            # that's always where HNM puts it before saving
            example["positive"] = example["positive"][0]
            return example
        dataset = dataset.map(_map_positive)
        dataset = dataset.remove_columns(["labels"])
    

    elif DATA_FORMAT == "apn_pairs":
        print("Formatting data as Anchor Positive/Negative Pairs")
        anchors = []
        pair_members = []
        labels = []

        # For each anchor code, the first element in the data row is the positive
        # example, so we know we can grab that and just label it with "1"
        for i in range(len(dataset)):
            anchors.append(dataset[i]["anchor"])
            pair_members.append(dataset[i]["positive"][0])
            labels.append(1)

            # Then, every remaining element in the row is a negative example found
            # with HNM, so we iterate through the list and label them all with "0"
            for j in range(1, NEGATIVES_PER_DATASET_ROW + 1):
                if j < len(dataset[i]["positive"]):
                    anchors.append(dataset[i]["anchor"])
                    pair_members.append(dataset[i]["positive"][j])
                    labels.append(0)

        dataset = dataset.from_dict({
            "anchor": anchors,
            "pair_member": pair_members,
            "label": labels
        })


    elif DATA_FORMAT == "scored_pairs":
        print("Formatting data as Scored Pairs")
        anchors = []
        pair_members = []
        scores = []
        # This process is the same as for apn-pairs, except the label is essentially
        # just replaced with a numeric, floating point score. Note that this means
        # for HNM outputs that produced labeled lists with `output_scores=False`, 
        # this format is actually equivalent to scored_pairs.
        for i in range(len(dataset)):
            anchors.append(dataset[i]["anchor"])
            pair_members.append(dataset[i]["positive"][0])
            scores.append(dataset[i]["labels"][0])
            for j in range(1, NEGATIVES_PER_DATASET_ROW + 1):
                if j < len(dataset[i]["positive"]):
                    anchors.append(dataset[i]["anchor"])
                    pair_members.append(dataset[i]["positive"][j])
                    scores.append(dataset[i]["labels"][j])

        dataset = dataset.from_dict({
            "anchor": anchors,
            "pair_member": pair_members,
            "score": scores
        })


    elif DATA_FORMAT == "apn_triplets": 
        print("Formatting data as APN-Triplets")
        # For uniqueness constraints, we can only get one triplet example per
        # row, so we only need the first negative
        def _map_triplet(example):
            if len(example["positive"]) > 1:
                example["negative"] = example["positive"][1]
            else:
                example["negative"] = None
            example["positive"] = example["positive"][0]
            return example

        dataset = dataset.map(_map_triplet)
        dataset = dataset.filter(lambda example: example["negative"] is not None)
        dataset = dataset.remove_columns(["scores"])


    elif DATA_FORMAT == "apn_list":
        print("Formatting data as APN List")
        # We need all rows to have the same numbers of negatives, so filter
        # out any with too few
        dataset = dataset.filter(
            lambda example: len(example["positive"]) >= (1 + NEGATIVES_PER_DATASET_ROW)
        )

        # For an APN list, we actually have to physically create a negative "key"
        # entry in the dataset dictionary, rather than just leave them all in a
        # list
        def _map_n_negatives(example):
            for j in range(1, NEGATIVES_PER_DATASET_ROW + 1):
                negative_key = "negative_" + str(j)
                example[negative_key] = example["positive"][j]
            example["positive"] = example["positive"][0]
            return example

        dataset = dataset.map(_map_n_negatives)
        dataset = dataset.remove_columns(["scores"])


    elif DATA_FORMAT == "labeled_list":
        # No actual processing to do because this is the format HNM data already
        # comes in
        print("HNM Data already formatted as labeled list, no processing needed")

# We're just loading from a text file for retriever training, much more 
# straightforward--no need to change formatting or anything like that   
else:
    examples = []

    print("Loading fine-tuning positive pairs...")
    with fs.open(FINE_TUNING_DATA_FILE) as fp:
        for line in fp:
            # Blob storage is bytes-based, so we need to decode before string operations
            line_str = line.decode("utf-8")
            if line_str.strip() != "":
                examples.append(line_str.strip())

    # Randomize the loaded examples while they're still in pair form, then sub-sample
    if RANDOM_SEED > 0:
        random.seed(RANDOM_SEED)
    random.shuffle(examples)
    examples = examples[:TRAINING_SIZE + VALIDATION_SIZE]

    # Sub-divide into anchor codes and nonstandard input codes
    anchor_codes = []
    positive_codes = []
    for ex in examples:
        pair = ex.split("|")
        anchor_codes.append(pair[0].strip())
        positive_codes.append(pair[1].strip())

    assert len(anchor_codes) == TRAINING_SIZE + VALIDATION_SIZE
    assert len(positive_codes) == TRAINING_SIZE + VALIDATION_SIZE

    print(f"{len(examples)} pairs loaded ({TRAINING_SIZE} training, {VALIDATION_SIZE} validation).")

    dataset = Dataset.from_dict({
        "anchor": anchor_codes,
        "positive": positive_codes
    })

# Now make the train-test split
dataset = dataset.train_test_split(test_size=VALIDATION_SIZE)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

## Step 3: Instantiate Model

Data preparation is the most complicated part of the fine-tuning process, so now that it's done, we just have to instantiate the model and define some training arguments. We'll fetch the model from remote, if necessary, then load it here with the appropriate `sentence-transformers` object (a `SentenceTransformer` for training the retriever, and a `CrossEncoder` for training the reranker). The choice of object is controlled by the setting `TRAINING_TRANSFORMER` at the top of the code. Set it to `True` if you're training a retriever, and `False` if you're training a reranker.

Note that while it's possible to set this value to `True` and then train a secondary Transformer to act as a pseud-reranker, we do not recommend this approach. Retrievers / Transformers always use the cosine similarity to score results. Since the first ANN search retriever already uses cosine similarity to score, using a second pass of cosine similarity--even if the embeddings that produce it are different--does not add a meaningful degree of new information. We want a true semantic comparison, not a similarity match.

If you're instantiating a `CrossEncoder`, some model parameters may be automatically logged by `sentence-transformers` as `MISSING` or `UNEXPECTED`. This behavior is completely normal. `CrossEncoders` are models that have an extra neural network layer on top of their existing architecture to "pool" hidden state values from the previous layers (essentially combining or averaging all the small numeric calculations into a single aggregate decisions). When creating a new `CrossEncoder` from scratch, these values will default to a random initial state, since they haven't been trained on the right way to average. The printed logs are just informing you of this fact, but after running fine-tuning, the message will disappear because the parameters will have been set appropriately.


In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import os

TRAINING_TRANSFORMER = False

MODEL_NAME = "intfloat_e5-large-v2_0.3_1e05"

# Make sure to set this directory appropriately based on where you're trying
# to fetch the model from. The `models/` directory has three sub-folders,
# `tsdae/`, `rerankers/`, and `fine_tuned/`.
TRAINED_DIR = "fine_tuned/"

if TRAINING_TRANSFORMER:
    # First, check if the model exists locally--if it does, nothing to do here
    if os.path.exists(MODEL_NAME):
        print("Model exists locally, loading it...")
    else:   
        if fs.exists("models/" + TRAINED_DIR + MODEL_NAME):
            print("Found trained model, loading from remote...")
            fs.get("models/" + TRAINED_DIR + MODEL_NAME, '.')
            print("Model loaded to local memory.")
        else:
            print("Could not find model at specified path.")
            print(
                "Check model name (esp. parameter numbers, underscores, and dashes) and fetch directory."
            )
        
    print("Instantiating sentence transformer...")
    model = SentenceTransformer(MODEL_NAME, )

# We're training a reranker
else:
    print("Instantiating reranker...")
    model = CrossEncoder(MODEL_NAME)

## Step 4: Instantiate Loss Evaluators

With the model instantiated, all that's left is to define our training arguments. There are four main hyper-parameters to set.

* _Learning Rate:_ This governs how quickly the model incorporates changes from the current batch of examples it's calculating into its overall prediction framework. Generally, this value should be quite low, somewhere between 1e-7 and 5e-5. We have found that a default setting of 1e-6 is extremely effective for most situations.
* _Number of Epochs:_ This determines how many passes the model will make through the entire set of training data. Fine-tuning LLMs are especially prone to overfitting, so this value should be very low most of the time. A default value of 1 is usually the right choice, though for reranker training, it could potentially be 2 or even 3 depending on experimental performance.
* _Batch Size:_ This governs how big of a training batch the model will consider at one time. Larger values will lead to slower training iterations, but generally more stable numeric performance (since larger values have a more accurate average gradient calculation, which leads to fewer oscillations across batches). We have found a default value of 64 to be performant for nearly all applications. The only time you may wish to increase the Batch Size is if training a reranker with multiple epochs, since the extra stability can improve results.
* _Loss Function:_ This choice is one that was largely already determined by training data format above, but it must be specified here in case there's more than one eligible choice. If you don't select a loss function compatible with your training format (see Step 2 for a discussion), then fine-tuning will not work.


In [ ]:
from sentence_transformers.losses import (
    MultipleNegativesRankingLoss,
    TripletLoss,
    ContrastiveLoss,
    OnlineContrastiveLoss,
    CoSENTLoss,
    AnglELoss
)
from sentence_transformers.cross_encoder.losses import MultipleNegativesRankingLoss as RerankerMNRL, LambdaLoss
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction, TranslationEvaluator
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from sentence_transformers.training_args import BatchSamplers

LEARNING_RATE = 1e-6
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 64
NUM_EVAL_BENCHMARKS = 10
BATCHES_PER_EVAL = int(len(train_dataset) / TRAIN_BATCH_SIZE / NUM_EVAL_BENCHMARKS)

# LOSS FUNCTION CONFIGURATION
# For SentenceTransformers models, options include:
#    mnrl, triplet, contrastive, online_contrastive, cosent, and angle
# For CrossEncoder models, options include:
#    mnrl, lambda
LOSS_FN_NAME = "lambda"
loss_margin = 1e-5

# MAKE SURE TO CHANGE THIS TO REFLECT ANY PROPERTIES THE MODEL USES
# e.g. account for the loss function, presence of HNM, data size, etc.
UPDATED_NAME = f"{MODEL_NAME.replace('/', '_')}_no_cn_hnm_{LOSS_FN_NAME}_tuned_{str(TRAINING_SIZE)}_{str(LEARNING_RATE).replace('-', '')}"


if TRAINING_TRANSFORMER:
    if LOSS_FN_NAME == "mnrl":
        loss = MultipleNegativesRankingLoss(model)
    elif LOSS_FN_NAME == "triplet":
        loss = TripletLoss(model, triplet_margin=loss_margin)
    elif LOSS_FN_NAME == "contrastive":
        loss = ContrastiveLoss(model, margin=loss_margin)
    elif LOSS_FN_NAME == "online_contrastive":
        loss = OnlineContrastiveLoss(model, margin=loss_margin)
    elif LOSS_FN_NAME == "cosent":
        loss = CoSENTLoss(model)
    elif LOSS_FN_NAME == "angle":
        loss = AnglELoss(model)

    args = SentenceTransformerTrainingArguments(
        # Required parameter:
        output_dir=None,
        # Optional training parameters:
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        warmup_ratio=0.1,
        fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
        bf16=True,  # Set to True if you have a GPU that supports BF16
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        # Optional tracking/debugging parameters:
        eval_strategy="steps",
        eval_steps=BATCHES_PER_EVAL,
        save_strategy="steps",
        save_steps=BATCHES_PER_EVAL,
        save_total_limit=2,
        logging_steps=BATCHES_PER_EVAL,
        run_name="fine_tuning",
    )

else:
    if LOSS_FN_NAME == "mnrl":
        loss = RerankerMNRL(model)
    elif LOSS_FN_NAME == "lambda":
        loss = LambdaLoss(model)

    args = CrossEncoderTrainingArguments(
        # Required parameter:
        output_dir=None,
        # Optional training parameters:
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
        bf16=True,  # Set to True if you have a GPU that supports BF16
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        # Optional tracking/debugging parameters:
        eval_strategy="steps",
        eval_steps=BATCHES_PER_EVAL,
        save_strategy="steps",
        save_steps=BATCHES_PER_EVAL,
        save_total_limit=2,
        logging_steps=BATCHES_PER_EVAL,
        run_name="ranker_tuning",
    )


## Step 5: Train the Model

With data loaded, a model instantiated, and hyper-parameter arguments defined, we can proceed directly to fine-tuning the model.

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.cross_encoder import CrossEncoderTrainer

if TRAINING_TRANSFORMER:
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=loss,
    )
else:
    trainer = CrossEncoderTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=loss,
    )

trainer.train()
model.save_pretrained(f"{UPDATED_NAME}/")

Optionally, we can also save this model to remote storage if we intend to analyze it or continue training in the future.

In [ ]:
import os
fs.put(UPDATED_NAME, f"/models/rerankers/{UPDATED_NAME}")